In [ ]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 2. Prepare Dummy Data (For demonstration purposes only)
# Note: If you have your own CSV file, replace this section 
# with: df = pd.read_csv('your_data.csv')
# ---------------------------------------------------------
np.random.seed(42) # Set seed for reproducibility
data_size = 1000

# Generate random physiological and activity data
data = {
    'Age': np.random.randint(18, 65, data_size),
    'Duration_min': np.random.uniform(10, 120, data_size),
    'Heart_Rate': np.random.uniform(80, 180, data_size),
    'Body_Temp_C': np.random.uniform(36.5, 39.5, data_size),
    'Weight_kg': np.random.uniform(50, 100, data_size),
}

df = pd.DataFrame(data)

# Create a dummy target variable 'Calories' based on a mathematical formula
# Calories = (Duration * 0.2) + (Heart Rate * 0.1) * (Weight * 0.05) + Random Noise
df['Calories'] = (df['Duration_min'] * 0.2) + (df['Heart_Rate'] * 0.1) * (df['Weight_kg'] * 0.05) + np.random.normal(0, 10, data_size)

print("First 5 rows of the dataset:")
print(df.head(), "\n")

# ---------------------------------------------------------
# 3. Separate Features (Inputs) and Target (Output)
# ---------------------------------------------------------
# 'Calories' is our target variable (y), everything else is a feature (X)
X = df.drop(columns=['Calories']) 
y = df['Calories']

# ---------------------------------------------------------
# 4. Train-Test Split (Splitting data for training and testing)
# ---------------------------------------------------------
# We use 80% of the data to train the model and 20% to test its performance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data size: {X_train.shape[0]} rows")
print(f"Testing data size: {X_test.shape[0]} rows\n")

# ---------------------------------------------------------
# 5. Data Scaling (Standardizing features to a common scale)
# ---------------------------------------------------------
# Scaling helps the model process variables with different units (e.g., Age vs. Heart Rate)
scaler = StandardScaler()

# Fit the scaler on the training data and transform it
X_train_scaled = scaler.fit_transform(X_train)
# Only transform the test data (do NOT fit, to prevent data leakage)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 6. Model Training
# ---------------------------------------------------------
print("Starting model training...")
# Initialize the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model using the scaled training data
model.fit(X_train_scaled, y_train)
print("Model trained successfully!\n")

# ---------------------------------------------------------
# 7. Model Evaluation (Testing the model's accuracy)
# ---------------------------------------------------------
# Predict calories for the unseen test data
y_pred = model.predict(X_test_scaled)

# Calculate performance metrics
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance Metrics ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} calories")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2 Score): {r2:.4f} (The closer to 1.0, the better)")

# ---------------------------------------------------------
# 8. Feature Importance (Identifying the most influential factors)
# ---------------------------------------------------------
# Extract the importance of each feature in predicting calories
feature_importances = pd.Series(model.feature_importances_, index=X.columns)

# Plot the top features
feature_importances.nlargest(10).plot(kind='barh', title='Feature Importance for Calories Burned', color='teal')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()